In [ ]:

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.utils.spark_session import spark
import pyspark.sql.functions
from pyspark.sql.functions import broadcast



In [ ]:
project_root = Path.cwd().parent


df_fact = spark.read.parquet(str(project_root / "output_dir/fact_requests")).sample(fraction=0.03, seed=42)
df_host = spark.read.parquet(str(project_root / "output_dir/dim_host"))
df_time = spark.read.parquet(str(project_root / "output_dir/dim_timestamp"))
df_status = spark.read.parquet(str(project_root / "output_dir/dim_status"))
df_endpoint = spark.read.parquet(str(project_root / "output_dir/dim_endpoint"))

df_fact.createOrReplaceTempView("fact_requests")
df_host.createOrReplaceTempView("host")
df_time.createOrReplaceTempView("time")
df_status.createOrReplaceTempView("status")
df_endpoint.createOrReplaceTempView("endpoint")


In [ ]:
spark.sql("""
    SELECT
        e.endpoint,
        COUNT(*) AS total,
        SUM(CASE WHEN s.status IN ('4xx', '5xx') THEN 1 ELSE 0 END) AS errors,
        SUM(CASE WHEN s.status IN ('4xx', '5xx') THEN 1 ELSE 0 END) / COUNT(*) AS error_rate
    FROM fact_requests f
    JOIN endpoint e ON f.endpt_key = e.endpt_key
    JOIN status s ON f.status_key = s.status_key
    GROUP BY e.endpoint
    ORDER BY error_rate DESC
    LIMIT 10
""").show()
# df_fact.printSchema()


26/06/29 22:49:03 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/29 22:49:04 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
[Stage 10:>                                                         (0 + 1) / 1]

In [ ]:
spark.sql("""
    SELECT /*+ BROADCAST(e) */
        e.extracted AS endpoint_category,
        PERCENTILE_APPROX(f.bytes, 0.95) AS p95_bytes
    FROM fact_requests f
    JOIN dim_endpoint e ON f.endpt_key = e.endpt_key
    GROUP BY e.extracted
""").show()